# DATA 622: Homework 7
**Author:** Brett Allen (ballen3@umbc.edu)

**Date Completed:** TBD

## Setup

In [54]:
!python -m pip install -q pandas requests beautifulsoup4 transformers scikit-learn nltk textblob chardet

### Imports

In [76]:
import requests
from bs4 import BeautifulSoup
import re
import zipfile
import os
import chardet
import nltk
import pandas as pd
import numpy as np
from transformers import pipeline
from textblob import TextBlob
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report

### Configurations

In [3]:
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /home/brett/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/brett/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## Questions
Extract the article of Sam Altman’s interview https://venturebeat.com/ai/sam-altman-at-ted-2025-inside-the-most-uncomfortable-and-important-ai-interview-of-the-year/.

In [4]:
url = "https://venturebeat.com/ai/sam-altman-at-ted-2025-inside-the-most-uncomfortable-and-important-ai-interview-of-the-year/"

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36'
}

response = requests.get(url, headers=headers)
response.status_code

200

In [5]:
# Parse with beautiful soup
soup = BeautifulSoup(response.text, "html.parser")

# Extract all paragraph text
paragraphs = soup.find_all("p")
article_text = " ".join([p.get_text() for p in paragraphs])

# Clean text
article_text = re.sub(r'\s+', ' ', article_text)

# Preview first 1000 characters
print(article_text[:1000] + '...') 

OpenAI CEO Sam Altman revealed that his company has grown to 800 million weekly active users and is experiencing "unbelievable" growth rates, during a sometimes tense interview at the TED 2025 conference in Vancouver last week. "I have never seen growth in any company, one that I've been involved with or not, like this," Altman told TED head Chris Anderson during their on-stage conversation. "The growth of ChatGPT — it is really fun. I feel deeply honored. But it is crazy to live through, and our teams are exhausted and stressed." The interview, which closed out the final day of TED 2025: Humanity Reimagined, showcased not just OpenAI's skyrocketing success but also the increasing scrutiny the company faces as its technology transforms society at a pace that alarms even some of its supporters. Altman painted a picture of a company struggling to keep up with its own success, noting that OpenAI's GPUs are "melting" due to the popularity of its new image generation features. "All day long

### 1. Use an LLM to summarize the article.

In [7]:
# Initialize LLM
llm = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    device_map="auto"
)

Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 338/338 [00:01<00:00, 174.74it/s]


In [9]:
prompt = f"""
Provide a summary for the following article. Only provide the summary and nothing else.

Article:
{article_text}
"""

result = llm(
    prompt,
    max_length=2048,
    num_return_sequences=1,
    do_sample=True,
    temperature=0.3
)

summary = result[0]["generated_text"]

Passing `generation_config` together with generation-related arguments=({'max_length', 'num_return_sequences', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [14]:
def substr_after(text: str, after: str) -> str:
    if after in text:
        idx = text.index(after)
        return text[idx+len(after):].strip()
    return text

In [15]:
# Ensure the prompt isn't included in the response
summary = substr_after(text=summary, after=prompt)

In [16]:
# Ensure "Summary" title isn't in the summary
summary = substr_after(text=summary, after="Summary:")

In [17]:
print("LLM Generated Summary:\n")
print(summary)

LLM Generated Summary:

This article discusses the rapid growth of OpenAI, a leading AI research company, and the ethical dilemmas surrounding its development. OpenAI has experienced tremendous growth since its inception, reaching 800 million weekly active users and facing intense scrutiny amid its rapidly expanding influence. Despite the company's efforts to maintain control, the sheer volume of users poses significant challenges, such as managing GPU resources and addressing potential risks associated with autonomous AI agents. OpenAI has also faced criticism for its handling of intellectual property issues, particularly concerning AI-generated artwork. The company's CEO, Sam Altman, acknowledges the immense power he holds but maintains that OpenAI continues to prioritize making AGI accessible and beneficial for humanity. Additionally, OpenAI is exploring various policy changes, including loosening restrictions on image generation models and shifting towards allowing users greater au

### 2. Identify the key topics, and the sentiment. Is the sentiment measured by the LLM different from one using a classification model such as Naïve Bayes or Support Vector Machine?

#### Key Topics (using LLM)

In [22]:
key_topics_prompt = f"""
List the key topics for the following article. Only provide the key topics and nothing else.

Article:
{article_text}
"""

result = llm(
    key_topics_prompt,
    max_length=2048,
    num_return_sequences=1,
    do_sample=True,
    temperature=0.3
)

key_topics = result[0]["generated_text"]

Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [23]:
# Ensure the prompt isn't included in the response
key_topics = substr_after(text=key_topics, after=key_topics_prompt)

In [25]:
# Ensure "Key Topics" title isn't in the summary
key_topics = substr_after(text=key_topics, after="Key Topics:")

In [26]:
print('LLM Generated Key Topics:\n')
print(key_topics)

LLM Generated Key Topics:

- OpenAI's growth and success
- Unprecedented growth rates
- ChatGPT and its impact
- Exponential growth challenges
- Social media competition
- Valuation and funding
- Policy changes
- Autonomous agents and AI safety
- Content moderation shifts
- Future of creativity and AI
- Copyright and intellectual property
- Safety and ethical considerations
- Corporate governance and decision-making
- Stakeholder engagement and criticism

These topics cover the main themes discussed in the article, including the rapid expansion of OpenAI, technological advancements, regulatory issues, and broader implications for society and the future of AI.


#### Sentiment Classification (using LLM)

In [35]:
sentiment_prompt = f"""
Analyze the sentiment of the following article.

Return ONLY one word: Positive, Negative, or Neutral.

Article:
{article_text}
"""

result = llm(
    sentiment_prompt,
    max_new_tokens=1,
    do_sample=False
)

llm_sentiment = result[0]["generated_text"]

Both `max_new_tokens` (=1) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [36]:
# Ensure the prompt isn't included in the response
llm_sentiment = substr_after(text=llm_sentiment, after=sentiment_prompt)

In [39]:
print('LLM Sentiment:\n')
print(llm_sentiment)

LLM Sentiment:

Neutral


#### Sentiment using Traditional ML

##### Obtain/Curate Sentiment Dataset (from Kaggle)
Dataset source: https://www.kaggle.com/datasets/abhi8923shriv/sentiment-analysis-dataset

In [42]:
extract_dir = "data"
zip_path = os.path.join(extract_dir, "sentiment-analysis-dataset.zip")
zip_path

'data/sentiment-analysis-dataset.zip'

In [40]:
%%bash
mkdir -p {extract_dir}
curl -L -o {zip_path}\
  https://www.kaggle.com/api/v1/datasets/download/abhi8923shriv/sentiment-analysis-dataset

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 54.4M  100 54.4M    0     0  30.0M      0  0:00:01  0:00:01 --:--:-- 36.6M


In [43]:
# Unzip with python to force overwrite if necessary
overwrite = True

with zipfile.ZipFile(zip_path, 'r') as z:
    if overwrite:
        z.extractall(extract_dir)
    else:
        for member in z.infolist():
            dest = os.path.join(extract_dir, member.filename)
            if not os.path.exists(dest):
                z.extract(member, extract_dir)

In [44]:
!du -sh {os.path.join(extract_dir, "*.csv")} 

464K	data/test.csv
72K	data/testdata.manual.2009.06.14.csv
4.5M	data/train.csv
139M	data/training.1600000.processed.noemoticon.csv


In [56]:
train_path = os.path.join(extract_dir, "train.csv")
test_path = os.path.join(extract_dir, "test.csv")
print(f'Train Path : {train_path}')
print(f'Test Path  : {test_path}')

Train Path : data/train.csv
Test Path  : data/test.csv


In [45]:
!head -5 {train_path}

textID,text,selected_text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (Km�),Density (P/Km�)
cb774db0d1," I`d have responded, if I were going","I`d have responded, if I were going",neutral,morning,0-20,Afghanistan,38928346,652860,60
549e992a42, Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative,noon,21-30,Albania,2877797,27400,105
088c60f138,my boss is bullying me...,bullying me,negative,night,31-45,Algeria,43851044,2381740,18
9642c003ef, what interview! leave me alone,leave me alone,negative,morning,46-60,Andorra,77265,470,164


**Observation:** Data is not UTF-8 encoded and needs to be converted before using it.

In [48]:
# Detect encoding
with open(train_path, "rb") as f:
    detected_encoding = chardet.detect(f.read())

print(detected_encoding)

{'encoding': 'hp-roman8', 'confidence': 0.8518033819371773, 'language': 'en'}


In [49]:
def convert_to_utf8(file_path: str) -> dict:
    if not os.path.exists(file_path):
        print(f'ERROR: File does not exist: {file_path}')
        return

    # Detect the original encoding
    with open(file_path, "rb") as f:
        detected_encoding = chardet.detect(f.read())

    encoding = detected_encoding["encoding"]

    # Read the contents of the file with the detected encoding
    with open(file_path, "r", encoding=encoding, errors="replace") as f:
        content = f.read()

    # Convert the contents to utf-8
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)

    # Verify conversion status
    with open(file_path, "rb") as f:
        converted_encoding = chardet.detect(f.read())

    # Return the status report
    return dict(
        original_encoding=detected_encoding,
        converted_encoding=converted_encoding,
    )

In [50]:
convert_to_utf8(train_path)

{'original_encoding': {'encoding': 'hp-roman8',
  'confidence': 0.8518033819371773,
  'language': 'en'},
 'converted_encoding': {'encoding': 'utf-8',
  'confidence': 0.8009804,
  'language': 'en'}}

In [51]:
convert_to_utf8(test_path)

{'original_encoding': {'encoding': 'hp-roman8',
  'confidence': 0.7762287141412714,
  'language': 'en'},
 'converted_encoding': {'encoding': 'utf-8',
  'confidence': 0.8004845,
  'language': 'en'}}

In [52]:
!head -5 {train_path}

textID,text,selected_text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (Kmý),Density (P/Kmý)
cb774db0d1," I`d have responded, if I were going","I`d have responded, if I were going",neutral,morning,0-20,Afghanistan,38928346,652860,60
549e992a42, Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative,noon,21-30,Albania,2877797,27400,105
088c60f138,my boss is bullying me...,bullying me,negative,night,31-45,Algeria,43851044,2381740,18
9642c003ef, what interview! leave me alone,leave me alone,negative,morning,46-60,Andorra,77265,470,164


In [53]:
!head -5 {test_path}

textID,text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (Kmý),Density (P/Kmý)
f87dea47db,Last session of the day  http://twitpic.com/67ezh,neutral,morning,0-20,Afghanistan,38928346,652860,60
96d74cb729, Shanghai is also really exciting (precisely -- skyscrapers galore). Good tweeps in China:  (SH)  (BJ).,positive,noon,21-30,Albania,2877797,27400,105
eee518ae67,"Recession hit Veronique Branquinho, she has to quit her company, such a shame!",negative,night,31-45,Algeria,43851044,2381740,18
01082688c6, happy bday!,positive,morning,46-60,Andorra,77265,470,164


**Observation:** Conversion to UTF-8 was successful.

In [58]:
# Load the train and test datasets
df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

In [61]:
df_train.shape, df_test.shape

((27481, 10), (4815, 9))

In [60]:
# Determine split ratio
total = len(df_train) + len(df_test)
print(f"Train : {len(df_train):,} ({len(df_train)/total:.1%})")
print(f"Test  :  {len(df_test):,} ({len(df_test)/total:.1%})")

Train : 27,481 (85.1%)
Test  :  4,815 (14.9%)


In [64]:
# Identify which column is in train but not in test
set(df_train.columns) ^ set(df_test.columns)

{'selected_text'}

In [65]:
df_train.head()

,textID,text,selected_text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (Kmý),Density (P/Kmý)
0,cb774db0d1,"I`d have responded, if I were going","I`d have responded, if I were going",neutral,morning,0-20,Afghanistan,38928346,652860.0,60
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative,noon,21-30,Albania,2877797,27400.0,105
2,088c60f138,my boss is bullying me...,bullying me,negative,night,31-45,Algeria,43851044,2381740.0,18
3,9642c003ef,what interview! leave me alone,leave me alone,negative,morning,46-60,Andorra,77265,470.0,164
4,358bd9e861,"Sons of ****, why couldn`t they put them on t...","Sons of ****,",negative,noon,60-70,Angola,32866272,1246700.0,26


In [66]:
df_test.head()

,textID,text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (Kmý),Density (P/Kmý)
0,f87dea47db,Last session of the day http://twitpic.com/67ezh,neutral,morning,0-20,Afghanistan,38928346.0,652860.0,60.0
1,96d74cb729,Shanghai is also really exciting (precisely -...,positive,noon,21-30,Albania,2877797.0,27400.0,105.0
2,eee518ae67,"Recession hit Veronique Branquinho, she has to...",negative,night,31-45,Algeria,43851044.0,2381740.0,18.0
3,01082688c6,happy bday!,positive,morning,46-60,Andorra,77265.0,470.0,164.0
4,33987a8ee5,http://twitpic.com/4w75p - I like it!!,positive,noon,60-70,Angola,32866272.0,1246700.0,26.0


In [71]:
# Determine representation of sentiment labels across the train dataset
sentiment_spread = df_train['sentiment'].value_counts()
sentiment_ratio = sentiment_spread / len(df_train)
pd.DataFrame({
    'Sentiment Spread': sentiment_spread,
    'Sentiment Ratio': sentiment_ratio,
    'Sentiment Percent': sentiment_ratio.map(lambda x: f'{x:.1%}')
})

,Sentiment Spread,Sentiment Ratio,Sentiment Percent
sentiment,,,
neutral,11118,0.404570,40.5%
positive,8582,0.312288,31.2%
negative,7781,0.283141,28.3%


In [72]:
# Drop rows with missing text or label
text_col = 'text'
label_col = 'sentiment'

train_clean = df_train[[text_col, label_col]].dropna()
test_clean = df_test[[text_col, label_col]].dropna()

In [73]:
# Separate text and label columns into feature/label dataset
X_train, y_train = train_clean[text_col], train_clean[label_col]
X_test, y_test = test_clean[text_col], test_clean[label_col]

##### Train Naive Bayes Model

In [74]:
nb_model = make_pipeline(CountVectorizer(), MultinomialNB())
nb_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('countvectorizer', ...), ('multinomialnb', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (strip_accents and lowercase) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


##### Evaluate Naive Bayes Model

In [77]:
y_pred_nb = nb_model.predict(X_test)
print("Naive Bayes Evaluation:\n")
print(classification_report(y_test, y_pred_nb))

Naive Bayes Evaluation:

              precision    recall  f1-score   support

    negative       0.68      0.60      0.64      1001
     neutral       0.60      0.69      0.64      1430
    positive       0.74      0.66      0.70      1103

    accuracy                           0.66      3534
   macro avg       0.67      0.65      0.66      3534
weighted avg       0.66      0.66      0.66      3534



##### Sentiment Classification (Naive Bayes)

In [78]:
nb_sentiment = nb_model.predict([article_text])[0]
print(f"Naive Bayes Sentiment: {nb_sentiment}")

Naive Bayes Sentiment: neutral


##### Train SVM

##### Evaluate SVM

##### Sentiment Classification (SVM)

### 3. What is the general emotion of the article?

### 4. What is the main theme of the article?